# Feedforward Meshing

The purpose of this notebook is to test different methods of feedforward point-cloud reconstruction. The primary methods tested so far are as follows:
- **VGGT-X:** transformer network with global aggregation module trained on intrinsic / extrinsic prediction
- **MapAnything:** builds on VGGT to impose metric scale reconstruction. Can take some degree of ground-truth data to improve the predictions / resultant pointcloud

Future improvements for integration:
- Loop closure (VGGT-Long / VGGT-SLAM)
- Sub-map alignment

Current testing to combine methods of frame sampling with mapanything, pushed into meshing. If this works will then integrate to improve the splatter module.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
from matplotlib import pyplot as plt
import numpy as np

import pyvista as pv

# Splatter module for information about video
from collab_splats.wrapper import Splatter
from collab_splats.utils.visualization import CAMERA_KWARGS, MESH_KWARGS, VIZ_KWARGS, visualize_splat

# Import optical flow module
from optical_flow import OpticalFlowFrameSelector, create_selection_summary_plot
from feedforward import Reconstructor

# Import MapAnything utils
from mapanything_utils import run_mapanything_pipeline

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Step 1: Load Configuration from Reconstructor

Load dataset configuration using Reconstructor's config system.

In [2]:
# Configuration paths
config_dir = Path("/workspace/collab-splats/docs/splats/configs")
dataset_name = "birds_date-02062024_video-C0043" #"bicycle_mapanything"

print(f"Loading configuration: {dataset_name}")
print(f"Config directory: {config_dir}")

# Load the splatter configuration
recon = Reconstructor.from_config_file(
    dataset=dataset_name,
    config_dir=config_dir,
)

print(f"\n" + "="*70)
print("Configuration Loaded")
print("="*70)
print(f"Dataset: {recon.config['file_path']}")
print(f"Input type: {recon.config['input_type']}")
print(f"Output: {recon.config['output_path']}")
print("="*70)

Loading configuration: birds_date-02062024_video-C0043
Config directory: /workspace/collab-splats/docs/splats/configs

Configuration Loaded
Dataset: /workspace/fieldwork-data/birds/2024-02-06/SplatsSD/C0043.MP4
Input type: video
Output: /workspace/fieldwork-data/birds/2024-02-06/environment/C0043


## Step 2: Frame Extraction with Optical Flow 

Use the optical flow method to find the optimal frames for video decimation

In [3]:
# Set parameters for optical flow frame selection
optical_flow_params = {
    'min_disparity': 50.0,
    'motion_weight': 0.5,
    'coverage_weight': 0.5,
    'rotation_threshold': 3.0,
    'adaptive_threshold': True,
    'verbose': True,

    # Process parameters (for selector.process_video)
    'selection_threshold': 0.51, 
}

# Use reconstruction class to extract frames (wraps optical flow)
recon.preprocess(
    use_optical_flow=True,
    # max_images=1000,
    optical_flow_kwargs=optical_flow_params,
    # overwrite=True,
)

⚠ Output directory already exists with 110 images: /workspace/fieldwork-data/birds/2024-02-06/environment/C0043/preproc/images
  Skipping preprocessing. Use overwrite=True to reprocess.


## Step 3: Pointcloud Creation via Forward-Pass

In [4]:
recon.setup_inference()

inference_kwargs = {
    # For model inference
    "memory_efficient_inference": True,
    "minibatch_size": 1,
    "force_rerun": True,

    # For model postprocessing
    "postprocess": True,
    "use_multiview_confidence": True, # Finds pixel-based overlap of depth across views
    "apply_mask": True,
    "mask_edges": True,
    "apply_confidence_mask": True,
    "confidence_percentile": 35.0,
}

# MapAnything contains the ability to decrease memory via minibatching
results = recon.infer(**inference_kwargs)

Auto-detected device: cuda
Setting up inference environment
Loading model: mapanything_v1
  Loading from HuggingFace: facebook/map-anything-v1
Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /workspace/models/hub/facebookresearch_dinov2_main


✓ Model loaded
Loading 110 images into memory
✓ Images loaded
✓ Ready for inference: 110 images
Running inference on 110 images
Applying post-processing...


Checking triangle intersections: 100%|██████████| 3/3 [00:00<00:00, 97.41it/s]



✓ Inference Complete
Performance:
  • Total time        : 5.88s
  • Time per frame    : 0.053s
  • Throughput (FPS)  : 18.71
  • Peak GPU memory   : 7.63 GB



## Step 4: Pointcloud refinement

In [ ]:
from sklearn.decomposition import PCA, IncrementalPCA
from tqdm.auto import tqdm

def create_feature_embeddings(results, n_components=16):
    """
    Project images into low-dimensional space using PCA.
    """
    # Setup feature extractor and grab image paths
    recon.setup_feature_extractor("samclip")
    image_paths = recon.config["image_paths"]

    # Ensure results and images are the same length
    assert (len(results) == len(image_paths))

    # Grab longest edge of images
    longest_edge = max(Image.open(image_paths[0]).size)

    # Get target H / W of images
    H, W = results[0]['mask'].squeeze().shape

    # Use smaller batch size to avoid memory spikes
    batch_size = min(5000, max(500, n_points // 20))  # More conservative batch size
    ipca = IncrementalPCA(n_components=n_components, batch_size=batch_size)

    for image_path in tqdm(image_paths, desc="Extracting features"):

        # Preprocess and extract features
        img_tensor = recon._feature_extractor.preprocess(image_path, resolution=longest_edge)
        img_batch = img_tensor.unsqueeze(0)
        feat = recon._feature_extractor(img_batch).squeeze(0)  # (D, H_p, W_p)

        # Upsample to size of image put through mapanything
        feat = F.interpolate(
            feat.unsqueeze(0),
            size=(H, W),
            mode="bilinear",
            align_corners=False,
        ).squeeze(0)  # (D, H, W)


    for image in tqdm(images, desc="Projecting images"):
        image = image.view(1, -1)
        ipca.partial_fit(image)

    for image in tqdm(images, desc="Transforming images"):
        image = image.view(1, -1)
        transformed = ipca.transform(image)
        transformed = transformed.view(image.shape[1], image.shape[2])
        yield transformed


longest_edge = max(Image.open(image_paths[0]).size)

image_paths = recon.config["image_paths"]

    img_tensor = recon._feature_extractor.preprocess(image_path, resolution=longest_edge)
    img_batch = img_tensor.unsqueeze(0)
    feat = recon._feature_extractor(img_batch).squeeze(0)  # (D, H_p, W_p)




In [6]:
from PIL import Image

image = Image.open(recon.config["image_paths"][0]).convert("RGB")

In [ ]:
import torch
from transformers import AutoModel
from torchvision.io import read_image

# Device setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Model Loading
model = AutoModel.from_pretrained("lorebianchi98/Talk2DINOv3-ViTB", trust_remote_code=True).to(device).eval()

[autoreload of timm.models.resnet failed: Traceback (most recent call last):
  File "/opt/conda/envs/nerfstudio/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/opt/conda/envs/nerfstudio/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 500, in superreload
    update_generic(old_obj, new_obj)
  File "/opt/conda/envs/nerfstudio/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 397, in update_generic
    update(a, b)
  File "/opt/conda/envs/nerfstudio/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 309, in update_function
    setattr(old, name, getattr(new, name))
ValueError: seresnext26tn_32x4d() requires a code object with 0 free vars, not 3
]
[autoreload of timm.models.efficientnet failed: Traceback (most recent call last):
  File "/opt/conda/envs/nerfstudio/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 276, in check


In [19]:

with torch.no_grad():
    text_embed = model.encode_text(["leaf"])
    image_embed = model.encode_image(image)

text_embed.shape, image_embed.shape

# normalize the features to perform cosine similarity
text_embed = text_embed / text_embed.norm(dim=-1, keepdim=True)
image_embed = image_embed / image_embed.norm(dim=-1, keepdim=True)

similarity = (image_embed @ text_embed.T).squeeze(0, -1).cpu().numpy()

In [24]:
from torch.nn import functional as F
texts = ["ground", "leaf", "tree", "background"]

with torch.no_grad():
    txt_embed = model.encode_text(texts)
    img_embed = model.encode_image([image])[0]

patches = F.normalize(img_embed, dim=-1)
text_out = F.normalize(txt_embed, dim=-1)
sims = txt_embed @ img_embed.transpose(0,1)

_, soft_masks = model.masker.sim2mask(sims, deterministic=True) 
soft_masks = torch.softmax(soft_masks / 0.001, dim=0)

img_size = 392
img = np.array(image).transpose(2, 0, 1)
img = F.interpolate(
    torch.tensor(img).unsqueeze(0).float(),
    size=(img_size, img_size),
    mode='bilinear',
    align_corners=False
).squeeze(0).numpy() / 255.0
masked_imgs = []
for i, text in enumerate(texts):
    mask = F.interpolate(
        soft_masks[i].view(1, 1, int(img_size/14), int(img_size/14)),
        size=(img_size, img_size),
        mode='bilinear',
        align_corners=False
    ).cpu().squeeze(0).numpy()
    masked_imgs.append(img * mask)

fig, ax = plt.subplots(1, len(texts)+1, figsize=(15,5))
ax[0].imshow(img.transpose(1,2,0))
ax[0].set_title("Image")
ax[0].axis('off')
for i, text in enumerate(texts):
    ax[i+1].imshow(masked_imgs[i].transpose(1,2,0))
    ax[i+1].set_title(f"'{text}'")
    ax[i+1].axis('off')
plt.show()

RuntimeError: shape '[1, 1, 28, 28]' is invalid for input of size 1369

In [ ]:
sim = recon._feature_extractor.compute_similarity(
    feat,
    positive=["tree"],
    negative=["sky", "leaf", "dirt", "ground"]
)

plt.imshow(sim.detach().cpu().numpy())


In [ ]:
from sklearn.decomposition import PCA, IncrementalPCA

# Use smaller batch size to avoid memory spikes
# batch_size = min(5000, max(500, n_points // 20))  # More conservative batch size
ipca = IncrementalPCA(n_components=16, batch_size=100000)

In [ ]:
import einops
# Flatten to (H * W, D)
flattened_featuers = einops.rearrange(feat, "d h w -> (h w) d")

In [ ]:
ipca.partial_fit(flattened_featuers.detach().cpu().numpy())

In [ ]:
recon.setup_feature_extractor("samclip")

In [ ]:
H, W = results[0]['mask'].squeeze().shape

In [ ]:
image = Image.open(image_paths[0])

inputs = processor(images=image, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.vision_model(**inputs, output_hidden_states=True)

In [ ]:
plt.imshow(sim.squeeze().detach().cpu().numpy())

In [ ]:
no_convert = np.asarray(Image.open(image_paths[0]))
convert = np.asarray(Image.open(image_paths[0]).convert("RGB"))

np.all(convert == no_convert)

In [ ]:
from tqdm import tqdm
from PIL import Image
from torch.nn import functional as F

image_paths = recon.config["image_paths"]
longest_edge = max(Image.open(image_paths[0]).size)


for image_path in tqdm(image_paths, desc="Extracting features"):
    img_tensor = recon._feature_extractor.preprocess(image_path, resolution=1024)
    img_batch = img_tensor.unsqueeze(0)
    feat = recon._feature_extractor(img_batch).squeeze(0)  # (D, H_p, W_p)

    # Upsample to size of image put through mapanything
    feat = F.interpolate(
        feat.unsqueeze(0),
        size=(H, W),
        mode="bilinear",
        align_corners=False,
    ).squeeze(0)  # (D, H, W)
    break

In [ ]:
plt.imshow(img_tensor.permute(1, 2, 0).detach().cpu().numpy())

In [ ]:
# Extract features for just this image
image_path = image_paths[view_idx]
extractor = self._feature_extractor

# Extract feature map for this single image
if isinstance(extractor, MaskCLIPExtractor):
    img_tensor = extractor.preprocess(image_path, resolution=feature_resolution)
    img_batch = img_tensor.unsqueeze(0)
    feat = extractor(img_batch).squeeze(0)  # (D, H_p, W_p)
    target_h, target_w = img_tensor.shape[-2], img_tensor.shape[-1]
    feat = F.interpolate(
        feat.unsqueeze(0),
        size=(target_h, target_w),
        mode="bilinear",
        align_corners=False,
    ).squeeze(0)  # (D, H, W)

In [ ]:
# Option 2: Get pointcloud data for custom processing
pointcloud = recon.feedforward_to_pointcloud(
    feature_extractor="samclip",  # or "clip-vit" 
    n_components=64,    
)

points = pointcloud['points']        # (N, 3) world coordinates
colors = pointcloud['colors']        # (N, 3) RGB colors
confidences = pointcloud['confidences']  # (N,) confidence value

In [ ]:
# Query the features
scores = recon.query_pointcloud(
    pointcloud, 
    positive=["tree", "leaf"],
    negative=["sky", "dirt", "ground"]
)

In [ ]:
point_cloud = pv.PolyData(pointcloud['points'])
point_cloud.point_data['RGB'] = pointcloud['colors']
print(f"  Points: {point_cloud.n_points:,}")

# Use collab_splats visualization
pcd_kwargs = MESH_KWARGS.copy()
pcd_kwargs.update({
    "point_size": 1,
    "render_points_as_spheres": True,
    "ambient": 0.3,
    "diffuse": 0.8,
    "specular": 0.1,
})

plotter = visualize_splat(
    mesh=point_cloud,
    mesh_kwargs=pcd_kwargs,
    viz_kwargs=VIZ_KWARGS,
)

plotter.show()

### Visualize selection metrics + frames

In [ ]:
# NOTE: `metrics` is not exposed by recon.preprocess().
# To use this plot, call OpticalFlowFrameSelector directly and capture
# its return value. Skipping for now.
#
# fig = create_selection_summary_plot(metrics, figsize=(16, 10))
# plt.show()


## Step 3: Use Feedforward Model

In [ ]:
image_dir = recon.config['output_path'] / "preproc" / "images"

pipeline_results = run_mapanything_pipeline(
    image_dir=image_dir,
    output_dir=recon.config['output_path'],
    cleanup_after=True,
    verbose=True,
    create_mesh=True,
)


Test piecewise pipeline

In [ ]:
from mapanything_utils import load_mapanything_model, load_and_preprocess_images

image_dir = recon.config['output_path'] / "preproc" / "images"
output_dir = recon.config['output_path']

# Setup directories
preproc_dir = output_dir / "preproc"
colmap_dir = preproc_dir / "colmap"

# Step 1: Load model
model = load_mapanything_model(model_name="facebook/map-anything", verbose=True)

# Step 2: Load images
views, image_paths = load_and_preprocess_images(image_dir, verbose=True)
image_names = [p.name for p in image_paths]


Forward pass through model

In [ ]:
from mapanything_utils import run_mapanything_inference

# Get model dimensions
model_width = views[0]['img'].shape[-1]
model_height = views[0]['img'].shape[-2]

# view_cut = views[-40:-20]

# Step 3: Run inference
outputs = run_mapanything_inference(
    model, 
    views, 
    verbose=True,
    apply_mask=True,
    apply_confidence_mask=True,
    use_multiview_confidence=False,
    confidence_percentile=35.0,
) #, **kwargs)

In [ ]:
for k, v in outputs.items():
    print (k, v.shape)

Now load the colmap reconstruction (rescaled) and see if we can mesh from that?

In [ ]:
import torch
from typing import List, Dict, Tuple, Any
from mapanything.utils.geometry import depthmap_to_world_frame

def format_outputs_for_visualization(
    outputs: List[Dict],
    filter_black_bg: bool = False,
    filter_white_bg: bool = False,
    masks: List[np.ndarray] = None,
) -> Tuple[Dict[str, np.ndarray], Any]:
    """
    Convert MapAnything model outputs to format for visualization.
    
    Args:
        outputs: List of prediction dictionaries from model.infer()
        views: List of view dictionaries from load_images()
        high_level_config: Configuration dictionary
        filter_black_bg: Whether to mask black background pixels
        filter_white_bg: Whether to mask white background pixels
    
    Returns:
        Tuple of (predictions dict, processed_data for visualization)
    """

    processed = []
    
    for i, pred in enumerate(outputs):
        # Extract tensors
        depth = pred["depth_z"][0].squeeze(-1)
        intrinsic = pred["intrinsics"][0]
        extrinsic = pred["camera_poses"][0]
        image = pred["img_no_norm"][0].cpu().numpy()
        
        # Compute world points
        pts3d_world, valid_depth = depthmap_to_world_frame(depth, intrinsic, extrinsic)
        
        # Build mask
        mask = pred["mask"][0].squeeze(-1).cpu().numpy().astype(bool) if "mask" in pred else np.ones_like(depth.cpu().numpy(), dtype=bool)
        mask = mask & valid_depth.cpu().numpy()
        
        # Background filtering
        if filter_black_bg or filter_white_bg:
            img_uint8 = (image * 255).astype(np.uint8)
            if filter_black_bg:
                mask = mask & (img_uint8.sum(axis=-1) >= 16)
            if filter_white_bg:
                mask = mask & ~((img_uint8 > 240).all(axis=-1))
        
        if masks is not None:
            _mask = cv2.resize(masks[i], (image.shape[1], image.shape[0]))
            mask = mask & _mask.astype(bool)
        
        processed.append({
            "world_points": pts3d_world.cpu().numpy(),
            "images": image,
            "extrinsic": extrinsic.cpu().numpy(),
            "intrinsic": intrinsic.cpu().numpy(),
            "final_mask": mask,
            "depth": depth.cpu().numpy(),
            "conf": pred["conf"][0].squeeze(-1).cpu().numpy(),
        })
    
    # Stack all arrays
    depth_stack = np.stack([p["depth"] for p in processed])
    predictions = {
        "world_points": np.stack([p["world_points"] for p in processed]),
        "images": np.stack([p["images"] for p in processed]),
        "extrinsic": np.stack([p["extrinsic"] for p in processed]),
        "intrinsic": np.stack([p["intrinsic"] for p in processed]),
        "final_mask": np.stack([p["final_mask"] for p in processed]),
        "depth": depth_stack[..., None],  # Add channel dimension
        "conf": np.stack([p["conf"] for p in processed]),
    }
    
    # Clean up GPU memory
    torch.cuda.empty_cache()
    
    return predictions

In [ ]:
# Run sky segmentation cells (Step 3.5) first to populate `masks`,
# or leave as None to skip sky masking.
if 'masks' not in dir():
    masks = None

preds = format_outputs_for_visualization(
    outputs=outputs,
    filter_black_bg=True,
    filter_white_bg=True,
    masks=masks,
)


In [ ]:
from mapanything.utils.hf_utils.viz import predictions_to_glb

glbscene = predictions_to_glb(
    preds.copy(),
    mask_black_bg=True,
    mask_white_bg=True,
    as_mesh=True,
    conf_percentile=35,
)

In [ ]:
import trimesh

scene = glbscene.dump(concatenate=False)

# 2. Remove tiny helper meshes (14 verts / 48 faces)
meshes = [
    g for g in scene
    if not (g.vertices.shape[0] == 14 and g.faces.shape[0] == 48)
]

print(f"Kept {len(meshes)} meshes")

# 4. Concatenate into one mesh
mesh = trimesh.util.concatenate(meshes)

# 5. Clean
mesh.merge_vertices()
mesh.remove_duplicate_faces()
mesh.remove_degenerate_faces()
mesh.remove_unreferenced_vertices()
mesh.remove_infinite_values()

In [ ]:
import open3d as o3d
import numpy as np
import trimesh

# Convert Trimesh -> Open3D
o3d_mesh = o3d.geometry.TriangleMesh(
    vertices=o3d.utility.Vector3dVector(mesh.vertices),
    triangles=o3d.utility.Vector3iVector(mesh.faces)
)

# Simplify / decimate to ~300k faces (or whatever target)
o3d_mesh = o3d_mesh.simplify_quadric_decimation(target_number_of_triangles=300_000)

# Convert back to Trimesh (if you want to keep using Trimesh)
mesh_simplified = trimesh.Trimesh(
    vertices=np.asarray(o3d_mesh.vertices),
    faces=np.asarray(o3d_mesh.triangles),
    process=False
)
print(mesh_simplified)

In [ ]:
mesh_simplified.show()

## Step 3.5: SkySegmetnation test

In [ ]:
import cv2
import numpy as np
import onnxruntime as ort
from huggingface_hub import hf_hub_download
import copy
import os

# Download the ONNX model from HuggingFace
model_path = hf_hub_download(
    repo_id="JianyuanWang/skyseg",
    filename="skyseg.onnx"
)

# Load the ONNX session
onnx_session = ort.InferenceSession(model_path)

# # Now use your existing functions
# image_path = "path/to/your/image.jpg"
# mask_filename = "output/masks/sky_mask.png"

# mask = segment_sky(image_path, onnx_session, mask_filename)

In [ ]:
import cv2
from mapanything.utils.hf_utils.viz import segment_sky
from tqdm import tqdm

masks_dir = preproc_dir / "masks"
masks_dir.mkdir(parents=True, exist_ok=True)

masks = []
for path in tqdm(image_paths):
    mask_filename = masks_dir / path.name.replace('.jpg', '_sky-mask.png')

    if mask_filename.exists():
        mask = cv2.imread(mask_filename).mean(-1)
    else:
        mask = segment_sky(path, onnx_session, mask_filename)
    masks.append(mask)
    # break

In [ ]:
img = views[0]['img'].detach().cpu().squeeze().permute(1, 2, 0).numpy()

plt.imshow(img)

In [ ]:
plt.imshow(mask)

## Step 4: Visualize PCD

In [ ]:
point_cloud = pv.PolyData(str(pipeline_results['point_cloud']))
print(f"  Points: {point_cloud.n_points:,}")

# Use collab_splats visualization
pcd_kwargs = MESH_KWARGS.copy()
pcd_kwargs.update({
    "point_size": 1,
    "render_points_as_spheres": True,
    "ambient": 0.3,
    "diffuse": 0.8,
    "specular": 0.1,
})

plotter = visualize_splat(
    mesh=point_cloud,
    mesh_kwargs=pcd_kwargs,
    viz_kwargs=VIZ_KWARGS,
)

plotter.show()